In [1]:
# Instalamos python-dotenv para poder cargar la API key de forma segura desde el archivo .env
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#KARENS_API_KEY

# Importamos las herramientas necesarias para cargar y leer las variables del archivo .env
from dotenv import load_dotenv
import os

# Cargamos las variables guardadas en el archivo .env
load_dotenv()

# Guardamos la API key en una variable para poder utilizarla más adelante
api_key = os.getenv("KARENS_API_KEY")

In [3]:
# Comprobamos que la API key se ha cargado correctamente sin mostrar su contenido
if api_key:
    print("API key cargada correctamente")
else:
    print("No se ha podido cargar la API key")

API key cargada correctamente


In [4]:
# Importamos requests para poder realizar peticiones HTTP a la API
import requests

# Definimos el endpoint de The Cat API que utilizaremos para consultar las razas
url = "https://api.thecatapi.com/v1/breeds"

In [5]:
# Creamos los headers para enviar nuestra API key de forma segura en la petición
headers = {
    "x-api-key": api_key
}

# Realizamos una petición GET al endpoint de razas
response = requests.get(url, headers=headers)

# Comprobamos el código de estado de la respuesta
print(response.status_code)

200


In [6]:
# Convertimos la respuesta de la API de formato JSON a un objeto de Python
data = response.json()

# Comprobamos qué tipo de objeto hemos recibido
print(type(data))

# Comprobamos cuántos registros hemos obtenido
print(len(data))

<class 'list'>
107


In [7]:
import pandas as pd

In [8]:
# Convertimos los datos obtenidos de la API en un DataFrame
df_cats = pd.DataFrame(data)

# Comprobamos las dimensiones del DataFrame
print(df_cats.shape)

(107, 18)


In [9]:
# Comprobamos la estructura del DataFrame y los tipos de datos de cada columna
df_cats.info()

<class 'pandas.DataFrame'>
RangeIndex: 107 entries, 0 to 106
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   id                  107 non-null    str   
 1   name                107 non-null    str   
 2   species_id          107 non-null    str   
 3   life_span           107 non-null    str   
 4   temperament         107 non-null    str   
 5   origin              100 non-null    str   
 6   country_codes       100 non-null    str   
 7   country_code        100 non-null    str   
 8   description         107 non-null    str   
 9   bred_for            0 non-null      object
 10  perfect_for         0 non-null      object
 11  breed_group         100 non-null    str   
 12  history             100 non-null    str   
 13  alt_names           55 non-null     str   
 14  reference_image_id  66 non-null     str   
 15  weight              107 non-null    object
 16  height              107 non-null    o

In [10]:
# Seleccionamos las columnas que nos interesan para el análisis
columns_analysis = [
    "name",
    "life_span",
    "temperament",
    "origin",
    "country_codes",
    "country_code",
    "description",
    "breed_group",
    "weight",
    "height"
]

# Creamos un nuevo DataFrame solo con las variables seleccionadas
df_cat_analysis = df_cats[columns_analysis].copy()

# Mostramos las primeras filas
df_cat_analysis.head()

,name,life_span,temperament,origin,country_codes,country_code,description,breed_group,weight,height
0,Abyssinian,14-17,"Active, Energetic, Independent, Intelligent, G...",Egypt,EG,EG,"Medium-sized, elegant cat with a distinctive t...",Short-haired,"{'imperial': '8-12', 'metric': '3.6-5.4'}","{'imperial': '10-12', 'metric': '25-30'}"
1,Aegean,9-12,"Affectionate, Social, Intelligent, Playful, Ac...",Greece,GR,GR,Medium-sized natural breed with a semi-longhai...,Semi-longhair,"{'imperial': '9-12', 'metric': '4-5.4'}","{'imperial': '9-11', 'metric': '23-28'}"
2,American Bobtail,11-15,"Intelligent, Interactive, Lively, Playful, Sen...",United States,US,US,Medium to large cat with a distinctive natural...,Short/Medium-hair,"{'imperial': '12-16', 'metric': '5.4-7.3'}","{'imperial': '11-14', 'metric': '28-36'}"
3,American Curl,12-16,"Affectionate, Curious, Intelligent, Interactiv...",United States,US,US,Medium-sized cat with distinctive backward-cur...,Short/Long-hair,"{'imperial': '7-11', 'metric': '3.2-5'}","{'imperial': '9-12', 'metric': '23-30'}"
4,American Ringtail,12-18,"Friendly, Affectionate, Intelligent, Active, P...",United States,US,US,Medium-sized cat with distinctive curled tail ...,Short/Long-hair,"{'imperial': '7-15', 'metric': '3.2-6.8'}","{'imperial': '9-12', 'metric': '23-30'}"


In [11]:
# Mostramos los diferentes valores que contiene la columna life_span
df_cat_analysis["life_span"].unique()

<StringArray>
['14-17',  '9-12', '11-15', '12-16', '12-18', '15-20', '14-18', '12-15',
 '12-14', '12-20', '14-20', '12-17', '15-18', '10-15', '10-14',     '1',
 '14-16',  '8-14',     '5', '13-16',  '9-15', '10-12',     '4', '11-16',
 '11-14', '14-15', '13-15', '11-18', '14-19']
Length: 29, dtype: str

In [12]:
# Buscamos registros de life_span que no contienen un rango
df_cat_analysis[
    ~df_cat_analysis["life_span"].str.contains("-")
][["name", "life_span"]]

,name,life_span
24,Caracat,1
40,Dwelf,5
42,Elf Cat,5
56,Kellas Cat,1
68,Napoleon Longhair,4
75,Pantherette,1
92,Singapura Tabby,1


In [18]:
# Creamos el filtro para los registros cuyo life_span no contiene un rango
life_span_single = ~df_cat_analysis["life_span"].str.contains("-")

# Mostramos información adicional de estos registros
df_cat_analysis.loc[
    life_span_single,
    [
        "name",
        "life_span",
        "origin",
        "description",

    ]
]

,name,life_span,origin,description
24,Caracat,1,NaN,"Large hybrid between caracal and domestic cat. Extremely rare and controversial.,16-22,41-56,14-20,36-51,20-30,9.1-13.6,15-25,6.8-11.3,15-20"
40,Dwelf,5,NaN,"Small, hairless cat with curled ears and short legs. Triple hybrid combining Sphynx, Munchkin, and American Curl.,6-9,15-23,5-8,13-20,4-9,1.8-4,3-7,1.4-3.2,12-15"
42,Elf Cat,5,NaN,"Medium-sized hairless cat with distinctive curled ears. Combines Sphynx and American Curl traits.,9-12,23-30,8-10,20-25,10-15,4.5-6.8,8-12,3.6-5.4,12-15"
56,Kellas Cat,1,NaN,"Large, melanistic wild-domestic hybrid found in Scotland. Not a true domestic breed.,14-18,36-46,12-16,30-41,8-15,3.6-6.8,6-12,2.7-5.4,Unknown"
68,Napoleon Longhair,4,NaN,"Small to medium cat with short legs and long plush coat. Longhaired variety of the Minuet.,7-9,18-23,6-8,15-20,5-9,2.3-4,4-7,1.8-3.2,9-15"
75,Pantherette,1,NaN,"Medium to large, muscular black cat bred to resemble a black panther. Experimental breed.,12-15,30-38,10-13,25-33,10-18,4.5-8.2,8-14,3.6-6.4,12-15"
92,Singapura Tabby,1,NaN,"Small, muscular cat with tabby pattern variation of Singapura. Extremely rare.,7-9,18-23,6-8,15-20,4-8,1.8-3.6,4-6,1.8-2.7,12-15"


In [17]:
# Mostramos el contenido completo de las columnas de texto
pd.set_option("display.max_colwidth", None)

In [19]:
# Creamos el filtro para los registros cuyo life_span no contiene un rango
life_span_prueba = df_cat_analysis["life_span"].str.contains("-")

# Mostramos información adicional de estos registros
df_cat_analysis.loc[
    life_span_prueba,
    [
        "name",
        "life_span",
        "origin",
        "description",

    ]
]

,name,life_span,origin,description
0,Abyssinian,14-17,Egypt,"Medium-sized, elegant cat with a distinctive ticked coat pattern giving a wild appearance. Known for its athletic build, almond-shaped eyes, and highly active, curious personality."
1,Aegean,9-12,Greece,"Medium-sized natural breed with a semi-longhaired coat, muscular build, and almond-shaped eyes. One of the oldest domesticated cat breeds, known for its love of water and fishing heritage."
2,American Bobtail,11-15,United States,"Medium to large cat with a distinctive naturally short tail, wild appearance, and athletic build. Known for its dog-like personality and exceptional intelligence."
3,American Curl,12-16,United States,"Medium-sized cat with distinctive backward-curling ears, elegant body, and silky coat. Known for its kitten-like personality that persists into adulthood."
4,American Ringtail,12-18,United States,Medium-sized cat with distinctive curled tail that curls over the back. Natural mutation first observed in California.
...,...,...,...,...
102,Toyger,12-15,United States,"Medium to large, muscular cat with distinctive tiger-like tabby stripes. Bred to resemble a miniature tiger."
103,Turkish Angora,12-18,Turkey,"Medium-sized, elegant cat with long silky coat and graceful build. One of the oldest natural cat breeds."
104,Turkish Van,12-17,Turkey,"Large, muscular cat with distinctive Van pattern (colored head and tail, white body). Famous for loving water."
105,Ukrainian Levkoy,12-15,Ukraine,"Medium-sized, hairless cat with distinctive folded ears. Unique combination of hairlessness and ear fold."


In [ ]:
# Se detectaron 7 registros con anomalías estructurales. Se compararon con los registros correctamente formateados para determinar si el problema afectaba únicamente a variables concretas o al registro completo.